# 1. Components

Components are the molecules that a structure is built from: lipids, surfactants, ions and solvents.

In this notebook you will:

1. see the assets in the Shapespyer library
2. load a component from the library and look at it
3. make a component from a file of your own
4. make a component from a SMILES string

Run the setup cell first. On Colab it installs Shapespyer, which takes about a minute. Locally it does nothing.

In [ ]:
# @title Setup: on Colab, install Shapespyer (about 1 min)
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q "shapespyer @ git+https://gitlab.com/simnavi/shapespyer.git@v0.3.0"

    # nglview, behind .show(), draws molecules through Colab's widget manager
    from google.colab import output
    output.enable_custom_widget_manager()

    # keep the library, Temp and Projects in /content, to browse them in the Files panel
    from shapes.project import Config
    config = Config()
    config.app_path = "/content/shapespyer"
    config.save()

In [ ]:
import urllib.request
from pathlib import Path

from shapes.basics.utils import LogConfiguration
from shapes.project import Library, Surfactant

LogConfiguration("WARNING")

library = Library()

### The library

The first `Library()` of a session reads the remote index of assets and downloads their files into `LocalAssets/` in the application directory. On Colab that is `/content/shapespyer/LocalAssets`: open the Files panel on the left to see it.

These are the assets ready to use, by type:

In [ ]:
library.available_assets

`find` tells where the files of an asset are. You can read these files, but don't change them: that would break the library.

In [ ]:
library.find("Surfactant", name="SDS")

### Load a component and look at it

`.show()` draws the molecule. It must be the last line of the cell.

In [ ]:
sds = Surfactant.load("SDS")
sds.show()

### A component from your own file

Here we download a `.gro` file, as if it were your own. The name of the component must match the residue name in the file.

On Colab you can also upload your own file: drag it into the Files panel, or run `from google.colab import files; files.upload()`.

In [ ]:
url = "https://gitlab.com/simnavi/shapespyer/-/raw/v0.3.0/assets/components/surfactants/CTA/CTA.gro"
my_file = Path("CTA.gro")
urllib.request.urlretrieve(url, my_file)

cta = Surfactant(name="CTA", paths=[my_file])
cta.show()

### A component from a SMILES string

Here we make SDS from its SMILES string. Hydrogens are added, and the atoms follow the SMILES order. We write the tail first, so the "bone" of the molecule runs from the tail end (`mint` is 0) to the head (`mext` is the last atom), as a micelle needs.

The name becomes the residue name in the file, at most 5 characters. The component gets a structure file only, with no topology. So it can be built into structures, but not equilibrated.

In [ ]:
sds_from_smiles = Surfactant.from_smiles(
    "SDS",
    "CCCCCCCCCCCCOS(=O)(=O)[O-]",
    box=(3.0, 3.0, 3.0),  # in nm
    formula="C12H25SO4",
    align_z=True,
)
print(sds_from_smiles.number_of_atoms, sds_from_smiles.mint, sds_from_smiles.mext)
sds_from_smiles.show()

### Clean up (optional)

This deletes the downloaded `CTA.gro`.

In [ ]:
my_file.unlink(missing_ok=True)
print(my_file.exists())